In [1]:
# Relevant packages for Stanford data visualization
import open3d as o3d
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import time

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
def plot_points(points, viewpoint, plot_viewpoint=True, 
                points_special_inds=None, extra_viewpoint=None, out_file="output.eps",
                save=True):
    # Prepare point clouds
    if points_special_inds is not None:
        scatter_points_special = go.Scatter3d(
            x=points[points_special_inds, 0], 
            y=points[points_special_inds, 1], 
            z=points[points_special_inds, 2], 
            mode='markers',
            marker=dict(
                size=0.7,
                opacity=1,
                color='green'
            )
        )

        scatter_points_normal = go.Scatter3d(
            x=np.delete(points[:, 0], points_special_inds), 
            y=np.delete(points[:, 1], points_special_inds), 
            z=np.delete(points[:, 2], points_special_inds), 
            mode='markers',
            marker=dict(
                size=0.7,
                opacity=1,
                color='red'
            )
        )
    else:
        scatter_points_normal = go.Scatter3d(
            x=points[:, 0], 
            y=points[:, 1], 
            z=points[:, 2], 
            mode='markers',
            marker=dict(
                size=0.7,
                opacity=1,
            )
        )
    # Store point clouds
    data = [scatter_points_normal]

    if points_special_inds is not None:
        data.append(scatter_points_special)

    # Create a Scatter3d object for the viewpoint
    if plot_viewpoint:
        scatter_viewpoint = go.Scatter3d(
            x=[viewpoint[0]], 
            y=[viewpoint[1]], 
            z=[viewpoint[2]], 
            mode='markers',
            marker=dict(
                size=10,
                color='blue',
                opacity=1
            )
        )
        data.append(scatter_viewpoint)

    if extra_viewpoint is not None:
        scatter_extra_viewpoint = go.Scatter3d(
            x=[extra_viewpoint[0]], 
            y=[extra_viewpoint[1]], 
            z=[extra_viewpoint[2]], 
            mode='markers',
            marker=dict(
                size=10,
                color='yellow',
                opacity=1
            )
        )
        data.append(scatter_extra_viewpoint)
        
    fig = go.Figure(data=data)


    # Update layout to remove background and axes
    fig.update_layout(
        autosize=False,
        width=500,
        height=500,
        margin=dict(
            l=0,  # left margin
            r=0,  # right margin
            b=0,  # bottom margin
            t=0,  # top margin
            pad=0  # padding
        ),
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            camera=dict(
                up=dict(x=0, y=1, z=0),
                center=dict(x=0, y=0, z=0),
                eye=dict(x=viewpoint[0], y=viewpoint[1], z=viewpoint[2])  # Adjust the eye position here
            )
        ),
        showlegend=False,
        plot_bgcolor='rgba(0,0,0,0)',
        paper_bgcolor='rgba(0,0,0,0)',
    )
    import random
    eye_x = random.uniform(-1, 1)
    eye_y = random.uniform(-1, 1)
    eye_z = random.uniform(-1, 1)
    
    # Update the figure layout
    fig.update_layout(scene_camera=dict(eye=dict(x=eye_x, y=eye_y, z=eye_z)))
    
    # # Show the plot
    fig.show()
    if save:
        pio.write_image(fig, out_file)

In [28]:
print("hej dp")
pcd = o3d.io.read_point_cloud("data/stanford/bunny/reconstruction/bun_zipper.ply")
from utils.visibility import visible_points, covisible_inds_np


points = np.asarray(pcd.points)
n_points = points.shape[0]
radius = 3.25


viewpoint0 = np.array([0, 0, -1.55])
#viewpoint1 = np.array([2, 0, -1.55])
viewpoint1 = np.array([0.5, 0, -1.55])
rot_hom = np.vstack((np.eye(3), np.zeros(3)))
T0 = np.append(rot_hom, np.append(viewpoint0, 1)[:, np.newaxis], axis=1)
T1 = np.append(rot_hom, np.append(viewpoint1, 1)[:, np.newaxis], axis=1)

visible_inds0 = visible_points(points, viewpoint0, hpr_radius=radius)
points_viewpoint0 = points[visible_inds0]
n_points0 = points_viewpoint0.shape[0]

visible_inds1 = visible_points(points, viewpoint1, hpr_radius=radius)
points_viewpoint1 = points[visible_inds1]
n_points1 = points_viewpoint1.shape[0]




points_permuted = np.swapaxes(points_viewpoint0, axis1=1, axis2=0)
points_homog = np.vstack((points_permuted, np.ones((n_points0))))
pc0 = np.swapaxes(np.matmul(np.linalg.inv(T0), points_homog)[:3], axis1=1, axis2=0)

points_permuted = np.swapaxes(points_viewpoint1, axis1=1, axis2=0)
points_homog = np.vstack((points_permuted, np.ones((n_points1))))
pc1_CS0 = np.swapaxes(np.matmul(np.linalg.inv(T0), points_homog)[:3], axis1=1, axis2=0)

pc_union = np.vstack((pc0, pc1_CS0))

visible_inds, visible_inds_pc0, visible_inds_pc1 = covisible_inds_np(pc0, pc_union, T0, T1, hpr_radius=radius)
pc_union_cov = pc_union[visible_inds]
n_points1_cov = pc_union_cov.shape[0]

points_permuted = np.swapaxes(pc_union_cov, axis1=1, axis2=0)
points_homog = np.vstack((points_permuted, np.ones((n_points1_cov))))
pc_union_cov_WCS = np.swapaxes(np.matmul(T0, points_homog)[:3], axis1=1, axis2=0)

#viewpoint = np.array([0, 0, -0.3])
pc_union_ground_truth_inds = np.intersect1d(visible_inds0, visible_inds1)
pc_union_ground_truth = points[pc_union_ground_truth_inds]
folder = 'images/point_clouds/stanford/'
plot_points(points_viewpoint0, viewpoint0, plot_viewpoint=False, out_file=folder + 'bunny_view0.eps')
plot_points(points_viewpoint1, viewpoint1, plot_viewpoint=False, out_file=folder + 'bunny_view1.eps')
plot_points(pc_union_ground_truth, viewpoint0, plot_viewpoint=False, out_file = folder + 'approx_gt_covisible.eps')
plot_points(pc_union_cov_WCS, viewpoint0, plot_viewpoint=False, out_file = folder + 'est_covisible.eps')

hej dp


In [3]:
import nuscenes as ns

# Init Nusc object
data_folder = 'data/nuscenes/'
version = 'v1.0-mini'
nusc = ns.nuscenes.NuScenes(version=version, dataroot=data_folder, verbose=False)

In [ ]:
# Some visualization on visibility on the separate Nuscens point clouds

from utils.data_handling import split_data
from utils.nuscenes_handling import read_nuscenes_data
from utils.visibility import visible_points, covisible_inds
import torch
# Set params
N = 1
T_close_thresh = 1.5

#
PC_scenes = read_nuscenes_data(nusc, downsample_factor=2, n_scenes=N, n_samples=N, T_close_thresh=T_close_thresh)
PC_scenes_training, PC_scenes_test = split_data(PC_scenes, scenes_training=N)

for i in range(len(PC_scenes_training)):
    PC_pair = PC_scenes_training[i][0]
    zero_vec = np.zeros((3))
    T0 = PC_pair.pose0
    T1 = PC_pair.pose1
    viewpoint1_CS0 = torch.matmul(torch.linalg.inv(T0), T1).cpu().numpy()[:3, 3]
    pc_in = PC_pair.PC0.pc.cpu().numpy()

    #
    print(f"\nScene: {i}")
    for radius in [3.25]:
        visible_inds = visible_points(pc_in, zero_vec, hpr_radius=radius)
        visible_inds_other = visible_points(pc_in, viewpoint1_CS0, hpr_radius=radius)
        visible_inds, visible_inds_pc0, visible_inds_pc1 = covisible_inds(PC_pair.PC0, PC_pair.PCUnion, T0, T1)
        # print(f"Percentage visible points own viewpoint vs other " +
        #       f"{np.around(100*len(visible_inds)/pc_in.shape[0], 3)} % vs "+
        #       f"{np.around(100*len(visible_inds_other)/pc_in.shape[0], 3)} %")
        print(f"Covisible point percentage in union point cloud {np.around(100*visible_inds.shape[0]/PC_pair.PCUnion.N_points,2)} %")
#plot_points(pc_in, zero_vec, points_special_inds=visible_inds, extra_viewpoint=viewpoint1_CS0)
pc_union = PC_pair.PCUnion.pc.cpu().numpy()
# pc_union = pc_union[~visible_inds]
plot_points(np.delete(pc_union, visible_inds, axis=0), viewpoint=zero_vec, plot_viewpoint=False)
plot_points(pc_union[visible_inds], viewpoint=zero_vec, plot_viewpoint=False)


In [32]:
# Some visualization on visibility on the joint Nuscens point clouds

from utils.data_handling import split_data
from utils.nuscenes_handling import read_nuscenes_data
from utils.visibility import covisible_inds, visible_points
import torch
import numpy as np
# Set params
N = 10
samples_per_scene = 10
T_close_thresh = 1.5
hpr_rads = np.arange(2, 5, 0.25)
#
PC_scenes = read_nuscenes_data(nusc, downsample_factor=1, n_scenes=N, n_samples=N*samples_per_scene, T_close_thresh=T_close_thresh)
PC_scenes_training, PC_scenes_test = split_data(PC_scenes, scenes_training=N)
# Initialize dictionaries to store the sum of percentages for each radius
sum_visible_perc_own = {rad: 0 for rad in hpr_rads}
sum_visible_perc_other = {rad: 0 for rad in hpr_rads}

for i in range(len(PC_scenes_training)):
    print(f"\nScene: {i}")
    for j in range(samples_per_scene):
      PC_pair = PC_scenes_training[i][j]
      zero_vec = np.zeros((3))
      T0 = PC_pair.pose0
      T1 = PC_pair.pose1
      viewpoint1_CS0 = torch.matmul(torch.linalg.inv(T0), T1).cpu().numpy()[:3, 3]
      PC_union = PC_pair.PCUnion.pc.cpu().numpy()
      pc = PC_pair.PC0.pc
      for hpr_radius in hpr_rads:
        #visible_inds, visible_inds_pc0, visible_inds_pc1 = covisible_inds(PC_pair.PC0, PC_pair.PCUnion, T0, T1, hpr_radius=hpr_radius)
        visible_inds = visible_points(pc, zero_vec, hpr_radius=hpr_radius)
        visible_inds_other = visible_points(pc, viewpoint1_CS0, hpr_radius=hpr_radius)
        # Calculate the percentage of visible points and add to the sum
        sum_visible_perc_own[hpr_radius] += 100*len(visible_inds)/pc.shape[0]
        sum_visible_perc_other[hpr_radius] += 100*len(visible_inds_other)/pc.shape[0]

      # plot_points(pc, viewpoint=zero_vec, plot_viewpoint=True, extra_viewpoint=viewpoint1_CS0, points_special_inds=visible_inds)

# Calculate and print average visibility for each radius
for hpr_radius in hpr_rads:
    avg_visible_perc_own = sum_visible_perc_own[hpr_radius] / len(PC_scenes_training)
    avg_visible_perc_other = sum_visible_perc_other[hpr_radius] / len(PC_scenes_training)
    print(f"Average percentage of visible points for rad {hpr_radius}: {np.around(avg_visible_perc_own, 2)}% vs other {np.around(avg_visible_perc_other, 2)}%")


#plot_points(pc, viewpoint=zero_vec, plot_viewpoint=True, extra_viewpoint=viewpoint1_CS0, points_special_inds=visible_inds)

We have collected 1 scenes
We have collected 2 scenes
We have collected 3 scenes
We have collected 4 scenes
We have collected 5 scenes
We have collected 50 number of samples
We have collected 6 scenes


We have collected 7 scenes
We have collected 8 scenes
We have collected 9 scenes
We have collected 10 scenes
Total number of samples: 100
Total number of scenes:  10

Scene: 0

Scene: 1

Scene: 2

Scene: 3

Scene: 4

Scene: 5

Scene: 6

Scene: 7

Scene: 8

Scene: 9
Average percentage of visible points for rad 2.0: 886.89% vs other 883.52%
Average percentage of visible points for rad 2.25: 919.85% vs other 915.62%
Average percentage of visible points for rad 2.5: 944.8% vs other 940.35%
Average percentage of visible points for rad 2.75: 963.7% vs other 959.55%
Average percentage of visible points for rad 3.0: 977.62% vs other 974.1%
Average percentage of visible points for rad 3.25: 987.18% vs other 984.38%
Average percentage of visible points for rad 3.5: 993.71% vs other 991.64%
Average percentage of visible points for rad 3.75: 997.29% vs other 995.89%
Average percentage of visible points for rad 4.0: 998.94% vs other 998.21%
Average percentage of visible points for rad 4.25: 999.61%

In [ ]:
%%timeit
visible_inds = visible_points(points, viewpoint, hpr_radius=3.25)

In [3]:
import matplotlib.pyplot as plt
import numpy as np

from visualization.point_e_tools import get_point_e_model, scatter_spheres
from visualization.point_e.point_e.util.plotting import plot_point_cloud

from utils.visibility import visible_points

In [4]:
# Generate synthetic point cloud with point-E (from text to point cloud)
# Takes about 1-2 min to run so don't re-run it if not necessary
samples, sampler = get_point_e_model(text='two fotballs')

creating base model...
creating upsample model...
downloading base checkpoint...
downloading upsampler checkpoint...


0it [00:00, ?it/s]

In [5]:
pc = sampler.output_to_point_clouds(samples)[0]
viewpoint = np.array([0.45, -0.4, 0.5])
visible_inds = visible_points(pc.coords, viewpoint, hpr_radius=3.7)
print(pc.coords.shape, type(pc.coords))
#pc.coords = pc.coords[visible_inds]
print(pc.coords.shape, type(pc.coords))


(4096, 3) <class 'numpy.ndarray'>
(4096, 3) <class 'numpy.ndarray'>


In [ ]:

#pc = sampler.output_to_point_clouds(samples)[0]
size = 0.50
fig = plot_point_cloud(pc, grid_size=1, fixed_bounds=((-size, -size, -size),(size, size, size)), grid_1d=True)
# Get the current Axes3D object
ax = plt.gca()
# Reduce the opacity of the point cloud
axes = fig.get_axes()
for ax in axes:
    for coll in ax.collections:
        coll.set_alpha(0.1)  # Set the alpha to 0.5 or any other value less than 1



# Add dotted lines from the point to the respective axes
#ax = scatter_spheres(x=-0.3, y=-0.4, z=0.3, ax=ax, col="r", size=size)
#ax = scatter_spheres(x=0.45, y=-0.4, z=0.5, ax=ax, col="b", size=size)
plt.show()

In [62]:
folder = 'images/point_clouds/point_e/'
scale = 1
viewpoint = np.array([scale, scale, scale])
# plot_points(pc.coords, viewpoint, plot_viewpoint=False, out_file=folder + 'fotballs.eps', save=False)
visible_inds = visible_points(pc.coords, viewpoint=viewpoint, hpr_radius=2.1)
plot_points(pc.coords, viewpoint, plot_viewpoint=True, out_file=folder + 'fotballs_visible_colored.eps', save=True,
            points_special_inds=visible_inds)